# 🔧 Predictive Maintenance — Synthetic Data Generator

Generates realistic **imbalanced** IoT sensor data simulating an industrial milling machine fleet.

### Key Design Decisions (based on real-world data)
- Target failure rate: **~3–5%** (matching UCI AI4I 2020 benchmark of 3.4%)
- Uses **probability-based** failure injection, not hard thresholds
- Being in a "danger zone" **increases** failure probability but does **not** guarantee failure
- Creates realistic class overlap (features don't perfectly separate)
- Produces models with realistic F1 scores (70–90%, not 99%)

### References
| Source | Failure Rate |
|--------|-------------|
| UCI AI4I 2020 Predictive Maintenance | 3.4% |
| NASA C-MAPSS Turbofan Degradation | ~2–4% |
| Industry benchmark (CNC / Milling) | 2–5% |

### Target Failure Breakdown (approximate)
| Mode | Description | Rate |
|------|-------------|------|
| TWF | Tool Wear Failure | ~0.4–0.8% |
| HDF | Heat Dissipation Failure | ~0.8–1.2% |
| PWF | Power Failure | ~0.5–1.0% |
| OSF | Overstrain Failure | ~0.5–1.0% |
| RNF | Random Failure | ~0.1–0.2% |
| **TOTAL** | **(with overlap)** | **~3–5%** |

In [1]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Imports & Environment Setup
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import pandas as pd
import numpy as np
import random
import os
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

print("All imports loaded successfully.")
print(f"   Pandas : {pd.__version__}")
print(f"   NumPy  : {np.__version__}")

All imports loaded successfully.
   Pandas : 2.2.3
   NumPy  : 2.2.6


## 1. Configuration
Fleet size, samples per machine, target failure rate, and per-mode rate targets. Everything downstream reads from these variables.

In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Configuration
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

N_MACHINES            = 10
SAMPLES_PER_MACHINE   = 5000
TARGET_FAILURE_RATE   = 0.035        # 3.5% — matching UCI AI4I benchmark
SEED                  = 42
TOTAL_SAMPLES         = N_MACHINES * SAMPLES_PER_MACHINE

np.random.seed(SEED)
random.seed(SEED)

# Per-mode target rates (sum > TARGET due to overlap between modes)
FAILURE_RATE_TARGETS = {
    'TWF': 0.006,   # ~0.6% Tool Wear Failure
    'HDF': 0.010,   # ~1.0% Heat Dissipation Failure
    'PWF': 0.008,   # ~0.8% Power Failure
    'OSF': 0.008,   # ~0.8% Overstrain Failure
    'RNF': 0.001,   # ~0.1% Random Failure
}

print(f"Fleet size            : {N_MACHINES} machines")
print(f"Samples per machine   : {SAMPLES_PER_MACHINE}")
print(f"Total records         : {TOTAL_SAMPLES:,}")
print(f"Target failure rate   : {TARGET_FAILURE_RATE:.1%}")
print(f"\nPer-mode targets:")
for mode, rate in FAILURE_RATE_TARGETS.items():
    print(f"  {mode}: {rate:.2%}")

Fleet size            : 10 machines
Samples per machine   : 5000
Total records         : 50,000
Target failure rate   : 3.5%

Per-mode targets:
  TWF: 0.60%
  HDF: 1.00%
  PWF: 0.80%
  OSF: 0.80%
  RNF: 0.10%


## 2. Machine Fleet Profiles
Each machine gets slightly different baseline characteristics — manufacturing variance means no two real machines are identical.

In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Build unique profile for every machine
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

machine_profiles = {}

for i in range(1, N_MACHINES + 1):
    machine_id = f"MILL-{i:03d}"
    machine_profiles[machine_id] = {
        'base_air_temp'       : np.random.uniform(293, 298),        # K
        'base_rot_speed'      : np.random.uniform(2600, 3000),      # RPM
        'base_torque'         : np.random.uniform(35, 45),          # Nm
        'base_vibration'      : np.random.uniform(0.3, 0.7),       # mm/s RMS
        'base_pressure'       : np.random.uniform(5.5, 6.5),       # bar
        'quality_type'        : np.random.choice(['L', 'M', 'H'], p=[0.5, 0.3, 0.2]),
        'age_years'           : np.random.uniform(0.5, 8.0),
        'maintenance_quality' : np.random.uniform(0.7, 1.0),       # 1.0 = perfect
    }

# Quick look
for mid, prof in machine_profiles.items():
    print(f"  {mid}  |  type={prof['quality_type']}  age={prof['age_years']:.1f}yr  maint_q={prof['maintenance_quality']:.2f}")

  MILL-001  |  type=L  age=0.9yr  maint_q=0.96
  MILL-002  |  type=L  age=1.9yr  maint_q=0.76
  MILL-003  |  type=L  age=2.7yr  maint_q=0.81
  MILL-004  |  type=L  age=5.1yr  maint_q=0.75
  MILL-005  |  type=L  age=5.6yr  maint_q=0.83
  MILL-006  |  type=M  age=2.8yr  maint_q=0.86
  MILL-007  |  type=H  age=5.0yr  maint_q=0.98
  MILL-008  |  type=L  age=6.7yr  maint_q=0.81
  MILL-009  |  type=H  age=6.3yr  maint_q=0.76
  MILL-010  |  type=L  age=3.2yr  maint_q=0.73


## 3. Generate Per-Machine Sensor Data
For each machine generate:
- **Timestamps** (1-min intervals with random gaps for sensor dropouts / maintenance windows)
- **Shifts** derived from hour of day
- **7 sensor channels:** Air Temp, Process Temp, Rotational Speed, Torque, Tool Wear, Vibration, Pressure

In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Generate complete sensor data for every machine
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print(f"[1/7] Generating data for {N_MACHINES} machines...\n")

machine_dfs = []

for machine_id, profile in machine_profiles.items():

    n = SAMPLES_PER_MACHINE

    # ────────────────────────────────────────
    # 3a. Timestamps
    # ────────────────────────────────────────
    start_date   = datetime(2024, 1, 1, 0, 0, 0)
    timestamps   = []
    current_time = start_date

    for _ in range(n):
        timestamps.append(current_time)
        interval = timedelta(minutes=1)

        # 0.5% chance of sensor dropout (5–30 min gap)
        if random.random() < 0.005:
            interval = timedelta(minutes=random.randint(5, 30))

        # 0.1% chance of maintenance window (2–8 hour gap)
        if random.random() < 0.001:
            interval = timedelta(hours=random.randint(2, 8))

        current_time += interval

    # ────────────────────────────────────────
    # 3b. Shifts
    # ────────────────────────────────────────
    shifts = []
    for ts in timestamps:
        hour = ts.hour
        if 6 <= hour < 14:
            shifts.append('Day')
        elif 14 <= hour < 22:
            shifts.append('Evening')
        else:
            shifts.append('Night')

    # ────────────────────────────────────────
    # 3c. Air Temperature (diurnal cycle)
    # ────────────────────────────────────────
    hours = np.array([ts.hour for ts in timestamps])
    diurnal_variation = 2 * np.sin(2 * np.pi * hours / 24)
    air_temp = (
        profile['base_air_temp']
        + diurnal_variation
        + np.random.normal(0, 1.5, n)
    )

    # ────────────────────────────────────────
    # 3d. Process Temperature
    # ────────────────────────────────────────
    process_temp = (
        air_temp
        + 10
        + np.random.normal(0, 0.8, n)
        + np.cumsum(np.random.normal(0, 0.002, n))   # slow heating trend
    )

    # ────────────────────────────────────────
    # 3e. Rotational Speed
    # ────────────────────────────────────────
    rot_speed = profile['base_rot_speed'] + np.random.normal(0, 80, n)

    # ────────────────────────────────────────
    # 3f. Torque (inversely related to speed)
    # ────────────────────────────────────────
    speed_deviation = (rot_speed - profile['base_rot_speed']) / profile['base_rot_speed']
    torque = (
        profile['base_torque']
        - speed_deviation * 15
        + np.random.normal(0, 8, n)
    )

    # ────────────────────────────────────────
    # 3g. Tool Wear — sawtooth with non-linear
    #     acceleration and periodic replacement
    # ────────────────────────────────────────
    tool_wear      = []
    current_wear   = 0
    wear_rate_base = random.uniform(2, 4)
    max_wear       = 200 + (1 - profile['maintenance_quality']) * 150

    for _ in range(n):
        wear_factor = 1.0 + (current_wear / max_wear) ** 2
        increment   = wear_rate_base * wear_factor * random.uniform(0.5, 1.5)
        current_wear += increment

        if current_wear > max_wear:
            current_wear   = 0
            wear_rate_base = random.uniform(2, 4)

        tool_wear.append(current_wear)

    tool_wear = np.array(tool_wear)

    # ────────────────────────────────────────
    # 3h. Vibration (mm/s RMS)
    #     Correlated with wear + random spikes
    # ────────────────────────────────────────
    wear_max    = tool_wear.max() if tool_wear.max() > 0 else 1
    wear_norm_v = tool_wear / wear_max

    vibration = (
        profile['base_vibration']
        + wear_norm_v * 2.5
        + np.random.normal(0, 0.15, n)
        + np.cumsum(np.random.normal(0, 0.0005, n))
    )

    spike_mask = np.random.random(n) < 0.003
    vibration[spike_mask] += np.random.uniform(2, 5, spike_mask.sum())
    vibration = np.maximum(vibration, 0.05)

    # ────────────────────────────────────────
    # 3i. Hydraulic Pressure (bar)
    #     Leak events = sustained drops
    # ────────────────────────────────────────
    wear_norm_p = tool_wear / wear_max

    pressure = (
        profile['base_pressure']
        + wear_norm_p * 0.8
        + np.random.normal(0, 0.1, n)
    )

    leak_starts = np.where(np.random.random(n) < 0.002)[0]
    for start in leak_starts:
        leak_duration  = random.randint(10, 50)
        end            = min(start + leak_duration, n)
        leak_magnitude = random.uniform(0.5, 2.0)
        pressure[start:end] -= np.linspace(0, leak_magnitude, end - start)

    pressure = np.clip(pressure, 3.0, 10.0)

    # ────────────────────────────────────────
    # 3j. Assemble single-machine DataFrame
    # ────────────────────────────────────────
    machine_df = pd.DataFrame({
        'Timestamp'        : timestamps,
        'Machine_ID'       : machine_id,
        'Type'             : profile['quality_type'],
        'Shift'            : shifts,
        'Air_Temp_K'       : np.round(air_temp, 2),
        'Process_Temp_K'   : np.round(process_temp, 2),
        'Rot_Speed_RPM'    : np.round(rot_speed, 1),
        'Torque_Nm'        : np.round(torque, 2),
        'Tool_Wear_Min'    : np.round(tool_wear, 1),
        'Vibration_mm_s'   : np.round(vibration, 3),
        'Pressure_bar'     : np.round(pressure, 2),
        'Machine_Age_Years': round(profile['age_years'], 1),
    })

    machine_dfs.append(machine_df)
    print(f"  ✓ {machine_id}: {len(machine_df)} records generated")

print(f"\n All {N_MACHINES} machines done.")

[1/7] Generating data for 10 machines...

  ✓ MILL-001: 5000 records generated
  ✓ MILL-002: 5000 records generated
  ✓ MILL-003: 5000 records generated
  ✓ MILL-004: 5000 records generated
  ✓ MILL-005: 5000 records generated
  ✓ MILL-006: 5000 records generated
  ✓ MILL-007: 5000 records generated
  ✓ MILL-008: 5000 records generated
  ✓ MILL-009: 5000 records generated
  ✓ MILL-010: 5000 records generated

 All 10 machines done.


## 4. Combine Fleet Data
Merge all individual machine DataFrames into one, sorted by timestamp.

In [5]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Combine & sort
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("[2/7] Combining fleet data...\n")

df = pd.concat(machine_dfs, ignore_index=True)
df = df.sort_values('Timestamp').reset_index(drop=True)

print(f" Combined fleet data: {len(df):,} rows × {df.shape[1]} cols")
df.head()

[2/7] Combining fleet data...

 Combined fleet data: 50,000 rows × 12 cols


,Timestamp,Machine_ID,Type,Shift,Air_Temp_K,Process_Temp_K,Rot_Speed_RPM,Torque_Nm,Tool_Wear_Min,Vibration_mm_s,Pressure_bar,Machine_Age_Years
0,2024-01-01,MILL-001,L,Night,295.37,305.26,3133.3,51.48,3.5,0.702,5.61,0.9
1,2024-01-01,MILL-009,H,Night,294.97,304.90,2794.3,47.72,5.2,0.474,5.57,6.3
2,2024-01-01,MILL-008,L,Night,293.01,302.97,2761.5,38.77,2.0,0.548,5.85,6.7
3,2024-01-01,MILL-007,H,Night,296.62,307.28,2759.1,61.91,1.1,0.473,6.38,5.0
4,2024-01-01,MILL-006,M,Night,292.29,301.90,2829.1,28.36,2.6,0.718,5.83,2.8


## 5. Probability-Based Failure Injection (Imbalanced)

Instead of hard thresholds, use **continuous risk scores** fed through a sigmoid, then sample Bernoulli trials.

### Why Probabilistic > Hard Thresholds

| Approach | Example | Problem |
|----------|---------|---------|
| **Hard threshold** | `if torque > 60 → FAILURE` | Deterministic, unrealistic, easy to classify |
| **Probabilistic** | `if torque > 60 → 8% chance of failure` | Stochastic, realistic, harder to classify |

### Pipeline
1. Compute **physics-based risk scores** (continuous 0→1) for each failure mode
2. **Calibrate** risk scores to match per-mode target rates
3. **Sample** Bernoulli trials — each record independently fails or not
4. **Aggregate** into overall failure label
5. **Post-hoc calibrate** if overall rate drifts outside ±1% of target

In [6]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Sigmoid helper
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# In reality, machine failure probability doesn't jump
# from 0 to 1 at a threshold — it increases gradually.
# Sigmoid models this smooth risk transition.

def sigmoid(x, center, steepness):
    return 1.0 / (1.0 + np.exp(-steepness * (x - center)))

print("Sigmoid helper defined.")

Sigmoid helper defined.


In [7]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Calculate physics-based RISK SCORES for each failure
#  mode. Each score is a continuous value between 0 and 1.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("[3/7] Injecting probability-based failures...")
print(f"  (Targeting ~{TARGET_FAILURE_RATE:.1%} failure rate)\n")

n = len(df)

# ── Derived physics quantities ───────────
temp_diff = df['Process_Temp_K'].values - df['Air_Temp_K'].values
power     = df['Torque_Nm'].values * df['Rot_Speed_RPM'].values * (2 * np.pi / 60)
strain    = df['Torque_Nm'].values * df['Tool_Wear_Min'].values

risk_scores = {}

# ──────────────────────────────────────────
# TWF — Tool Wear Failure
# Physics: Tool material fatigues after prolonged use.
# Risk increases exponentially as wear approaches limit.
# High vibration accelerates micro-crack propagation.
# ──────────────────────────────────────────
wear_risk = sigmoid(df['Tool_Wear_Min'].values, 220, 0.03)
vib_risk  = sigmoid(df['Vibration_mm_s'].values, 3.0, 1.5)
risk_scores['TWF'] = wear_risk * vib_risk

# ──────────────────────────────────────────
# HDF — Heat Dissipation Failure
# Physics: If heat generated > heat dissipated, thermal
# runaway occurs. Low temp differential + low airflow
# (low speed) = poor heat removal.
# ──────────────────────────────────────────
temp_risk  = sigmoid(-temp_diff, -8.6, 1.0)
speed_risk = sigmoid(-df['Rot_Speed_RPM'].values, -2700, 0.008)
risk_scores['HDF'] = temp_risk * speed_risk

# ──────────────────────────────────────────
# PWF — Power Failure
# Physics: Operating outside the safe power envelope.
# Both underpowered (stalling) and overpowered (burnout)
# are dangerous.
# ──────────────────────────────────────────
low_power_risk  = sigmoid(-power, -4000, 0.002)
high_power_risk = sigmoid(power, 18000, 0.0003)
risk_scores['PWF'] = np.maximum(low_power_risk, high_power_risk)

# ──────────────────────────────────────────
# OSF — Overstrain Failure
# Physics: Combined stress from high torque on a worn tool.
# Product quality affects material tolerance:
#   L → fails under less strain
#   M → moderate tolerance
#   H → highest tolerance
# ──────────────────────────────────────────
type_centers = df['Type'].map({'L': 11000, 'M': 13000, 'H': 15000}).values.astype(float)

osf_risk = np.zeros(n)
for i in range(n):
    osf_risk[i] = sigmoid(strain[i], type_centers[i], 0.0004)

risk_scores['OSF'] = osf_risk

# ──────────────────────────────────────────
# RNF — Random Failure
# Physics: Unpredictable component failure (electrical
# short, seal rupture, etc.). Slightly higher probability
# for older machines.
# ──────────────────────────────────────────
age_factor = df['Machine_Age_Years'].values / 10.0
risk_scores['RNF'] = np.full(n, 0.001) + age_factor * 0.0005

print(" Risk scores computed for all 5 failure modes.")
for mode, scores in risk_scores.items():
    print(f"  {mode}  mean_risk={scores.mean():.6f}  max_risk={scores.max():.4f}")

[3/7] Injecting probability-based failures...
  (Targeting ~3.5% failure rate)

 Risk scores computed for all 5 failure modes.
  TWF  mean_risk=0.032380  max_risk=0.5645
  HDF  mean_risk=0.062933  max_risk=0.5959
  PWF  mean_risk=0.159431  max_risk=0.9938
  OSF  mean_risk=0.069403  max_risk=0.8644
  RNF  mean_risk=0.001201  max_risk=0.0013


In [8]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Calibrate risk scores → sample Bernoulli failures
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

failure_cols = []

for mode, risk in risk_scores.items():
    target_rate = FAILURE_RATE_TARGETS[mode]
    col_name    = f'Failure_{mode}'

    # We want: mean(risk × calibration_factor) ≈ target_rate
    mean_risk = risk.mean()
    if mean_risk > 0:
        calibration = target_rate / mean_risk
        calibration = min(calibration, 10.0)       # cap to prevent prob > 1
    else:
        calibration = 0

    # Calibrated failure probability per record
    failure_prob = np.clip(risk * calibration, 0, 0.95)

    # Sample Bernoulli — each record independently fails or not
    df[col_name] = (np.random.random(n) < failure_prob).astype(int)
    failure_cols.append(col_name)

# ── Aggregate — failure if ANY mode triggered ─
df['Machine_Failure'] = df[failure_cols].max(axis=1)

# ── Human-readable failure mode string ────────
failure_mode_list = []
for _, row in df[failure_cols].iterrows():
    modes = [col.replace('Failure_', '') for col in failure_cols if row[col] == 1]
    failure_mode_list.append(','.join(modes) if modes else 'None')

df['Failure_Mode'] = failure_mode_list

failure_rate   = df['Machine_Failure'].mean()
n_failures     = int(df['Machine_Failure'].sum())
n_normal       = len(df) - n_failures
imbalance_ratio = n_normal / n_failures if n_failures > 0 else float('inf')

print(f" Failures sampled (before calibration):")
print(f"  Failure rate     : {failure_rate:.2%}")
print(f"  Normal           : {n_normal:,}")
print(f"  Failures         : {n_failures:,}")
print(f"  Imbalance ratio  : {imbalance_ratio:.1f}:1\n")
for fc in failure_cols:
    rate  = df[fc].mean()
    count = int(df[fc].sum())
    print(f"    {fc}: {rate:.3%} ({count:,} events)")

 Failures sampled (before calibration):
  Failure rate     : 3.26%
  Normal           : 48,372
  Failures         : 1,628
  Imbalance ratio  : 29.7:1

    Failure_TWF: 0.590% (295 events)
    Failure_HDF: 0.984% (492 events)
    Failure_PWF: 0.806% (403 events)
    Failure_OSF: 0.812% (406 events)
    Failure_RNF: 0.108% (54 events)


## 6. Post-Hoc Calibration
If the physics-based injection overshoots the target, randomly remove some failure labels. If it undershoots, randomly add some in high-risk zones (weighted by tool wear for realism). This ensures the dataset matches real-world imbalance regardless of exact sensor distributions.

In [10]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Post-hoc calibration — force overall rate into
#  TARGET ± 1% tolerance band
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

tolerance   = 0.01
actual_rate = df['Machine_Failure'].mean()
lower_bound = TARGET_FAILURE_RATE - tolerance
upper_bound = TARGET_FAILURE_RATE + tolerance

print(f"Actual rate before calibration: {actual_rate:.2%}")
print(f"Acceptable band: [{lower_bound:.2%}, {upper_bound:.2%}]")

if actual_rate > upper_bound:
    # ── Too many failures — randomly remove some ──
    excess           = actual_rate - TARGET_FAILURE_RATE
    failure_indices  = df[df['Machine_Failure'] == 1].index.tolist()
    n_to_remove      = int(excess * len(df))
    n_to_remove      = min(n_to_remove, len(failure_indices))

    if n_to_remove > 0:
        remove_indices = np.random.choice(failure_indices, size=n_to_remove, replace=False)
        df.loc[remove_indices, 'Machine_Failure'] = 0
        for fc in failure_cols:
            df.loc[remove_indices, fc] = 0
        df.loc[remove_indices, 'Failure_Mode'] = 'None'
        print(f"  ↓ Removed {n_to_remove:,} excess failure labels.")

elif actual_rate < lower_bound:
    # ── Too few failures — add some in high-risk zones ──
    deficit        = TARGET_FAILURE_RATE - actual_rate
    normal_indices = df[df['Machine_Failure'] == 0].index.tolist()
    n_to_add       = int(deficit * len(df))
    n_to_add       = min(n_to_add, len(normal_indices))

    if n_to_add > 0:
        # Weight by tool wear (higher wear = more likely to fail)
        weights = df.loc[normal_indices, 'Tool_Wear_Min'].values + 1   # avoid zero
        weights = weights / weights.sum()

        add_indices = np.random.choice(normal_indices, size=n_to_add,
                                       replace=False, p=weights)

        for idx in add_indices:
            chosen_mode = np.random.choice(failure_cols)
            df.loc[idx, 'Machine_Failure'] = 1
            df.loc[idx, chosen_mode]       = 1
            mode_name = chosen_mode.replace('Failure_', '')
            current   = df.loc[idx, 'Failure_Mode']
            if current == 'None':
                df.loc[idx, 'Failure_Mode'] = mode_name
            else:
                df.loc[idx, 'Failure_Mode'] = f"{current},{mode_name}"

        print(f"Added {n_to_add:,} failure labels in high-risk zones.")

else:
    print("Already within tolerance — no adjustment needed.")

# ── Recompute stats ──────────────────────
failure_rate    = df['Machine_Failure'].mean()
n_failures      = int(df['Machine_Failure'].sum())
n_normal        = len(df) - n_failures
imbalance_ratio = n_normal / n_failures if n_failures > 0 else float('inf')

print(f"\n After calibration:")
print(f"  Failure rate     : {failure_rate:.2%}")
print(f"  Normal           : {n_normal:,}")
print(f"  Failures         : {n_failures:,}")
print(f"  Imbalance ratio  : {imbalance_ratio:.1f}:1")

Actual rate before calibration: 3.26%
Acceptable band: [2.50%, 4.50%]
  Already within tolerance — no adjustment needed.

 After calibration:
  Failure rate     : 3.26%
  Normal           : 48,372
  Failures         : 1,628
  Imbalance ratio  : 29.7:1


## 7. Pre-Failure Degradation Signals
Real machines do not fail instantly, there are warning signs in the preceding readings:
- Vibration increases gradually
- Temperature creeps up
- Pressure drops slightly

This makes the classification problem more realistic and rewards models that can detect **early degradation patterns**.

In [11]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Inject subtle degradation in the readings BEFORE
#  each failure event (window = 20 readings)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("[4/7] Adding pre-failure degradation signals...\n")

DEGRADATION_WINDOW = 20
failure_indices    = df[df['Machine_Failure'] == 1].index.tolist()

for fail_idx in failure_indices:
    start = max(0, fail_idx - DEGRADATION_WINDOW)
    if start == fail_idx:
        continue

    degradation_range = range(start, fail_idx)
    n_degrade         = len(degradation_range)

    # Degradation intensity increases closer to failure (0 → 1)
    intensity = np.linspace(0, 1, n_degrade)

    for i, idx in enumerate(degradation_range):
        if idx in df.index:
            # Subtle vibration increase
            df.loc[idx, 'Vibration_mm_s'] += intensity[i] * np.random.uniform(0.1, 0.5)

            # Subtle temperature increase
            df.loc[idx, 'Process_Temp_K'] += intensity[i] * np.random.uniform(0.2, 0.8)

            # Subtle pressure drop
            df.loc[idx, 'Pressure_bar']   -= intensity[i] * np.random.uniform(0.05, 0.2)

print(f" Degradation applied to {len(failure_indices):,} failure events")
print(f"  Window: {DEGRADATION_WINDOW} readings before each failure")

[4/7] Adding pre-failure degradation signals...

  ✓ Degradation applied to 1,628 failure events
  ✓ Window: 20 readings before each failure


## 8. Inject Realistic Missing Values (~2%)
Real IoT sensors experience communication dropouts, sensor malfunctions, and data corruption during transmission. Simulate:
- **Per-sensor** independent dropout probability
- **Full-row** communication failures (rarer)

In [12]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Missing value injection
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("[5/7] Injecting realistic missing values (~2%)...\n")

missing_rate = 0.02

sensor_cols = [
    'Air_Temp_K', 'Process_Temp_K', 'Rot_Speed_RPM',
    'Torque_Nm', 'Tool_Wear_Min', 'Vibration_mm_s', 'Pressure_bar'
]

for col in sensor_cols:
    col_missing_rate = missing_rate * random.uniform(0.5, 1.5)
    mask = np.random.random(len(df)) < col_missing_rate
    df.loc[mask, col] = np.nan

# Occasional full-row dropout (communication failure)
row_dropout_mask = np.random.random(len(df)) < 0.001
df.loc[row_dropout_mask, sensor_cols] = np.nan

total_missing = df.isnull().sum().sum()
print(f"Missing values injected: {total_missing:,} total NaNs\n")
print(df.isnull().sum())

[5/7] Injecting realistic missing values (~2%)...

Missing values injected: 6,949 total NaNs

Timestamp               0
Machine_ID              0
Type                    0
Shift                   0
Air_Temp_K           1156
Process_Temp_K        653
Rot_Speed_RPM         698
Torque_Nm            1271
Tool_Wear_Min         640
Vibration_mm_s       1451
Pressure_bar         1080
Machine_Age_Years       0
Failure_TWF             0
Failure_HDF             0
Failure_PWF             0
Failure_OSF             0
Failure_RNF             0
Machine_Failure         0
Failure_Mode            0
dtype: int64


## 9. Inject Sensor Outliers (~0.5%)
- **Spike outliers** in temperature sensors (electrical interference)
- **Stuck sensor readings** — same value repeating for several consecutive rows

In [13]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Outlier injection
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("[6/7] Injecting sensor outliers (~0.5%)...\n")

outlier_rate = 0.005

# Temperature spikes
for col in ['Air_Temp_K', 'Process_Temp_K']:
    mask            = np.random.random(len(df)) < outlier_rate
    spike_direction = np.random.choice([-1, 1], mask.sum())
    spike_magnitude = np.random.uniform(15, 40, mask.sum())
    df.loc[mask, col] = df.loc[mask, col] + spike_direction * spike_magnitude

# Stuck sensor readings
stuck_starts = np.where(np.random.random(len(df)) < 0.0005)[0]
for start in stuck_starts:
    duration = random.randint(5, 20)
    end      = min(start + duration, len(df))
    col      = random.choice(['Rot_Speed_RPM', 'Torque_Nm', 'Vibration_mm_s'])
    if start < len(df):
        df.loc[start:end, col] = df.loc[start, col]

print(f"Outliers injected — {len(stuck_starts)} stuck-sensor events")

[6/7] Injecting sensor outliers (~0.5%)...

Outliers injected — 30 stuck-sensor events


## 10. Save to CSV & Generation Metadata

In [14]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Save
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("[7/7] Saving...\n")

output_filename = 'sensor_data.csv'
df.to_csv(output_filename, index=False)
file_size_mb = os.path.getsize(output_filename) / (1024 * 1024)

print(f"Saved to '{output_filename}' — {file_size_mb:.1f} MB")

[7/7] Saving...

Saved to 'sensor_data.csv' — 5.6 MB


In [18]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Generation metadata
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

metadata = {
    'total_records'       : len(df),
    'n_machines'          : N_MACHINES,
    'samples_per_machine' : SAMPLES_PER_MACHINE,
    'n_features'          : len(df.columns),
    'failure_rate'        : failure_rate,
    'n_failures'          : n_failures,
    'n_normal'            : n_normal,
    'imbalance_ratio'     : imbalance_ratio,
    'missing_values'      : total_missing,
    'target_failure_rate' : TARGET_FAILURE_RATE,
    'date_range'          : f"{df['Timestamp'].min()} to {df['Timestamp'].max()}",
    'failure_breakdown'   : {
        fc: {'rate': float(df[fc].mean()), 'count': int(df[fc].sum())}
        for fc in failure_cols
    },
    'machines'            : list(machine_profiles.keys()),
}

print("=" * 65)
print("  GENERATION COMPLETE")
print("=" * 65)
print(f"""
  Dataset Shape     : {df.shape}
  Machines          : {N_MACHINES}
  Date Range        : {metadata['date_range']}

  ┌──────────────────────────────────────────────┐
  │  CLASS DISTRIBUTION (IMBALANCED)             │
  ├──────────────────────────────────────────────┤
  │  Normal  (0): {n_normal:>8,}  ({1 - failure_rate:>6.2%})             │
  │  Failure (1): {n_failures:>8,}  ({failure_rate:>6.2%})             │
  │  Ratio      : {imbalance_ratio:>8.1f}:1                     │
  ├──────────────────────────────────────────────┤
  │  UCI AI4I 2020 Reference: 3.4% failure rate  │
  └──────────────────────────────────────────────┘

  Failure Mode Breakdown:
""")
for fc in failure_cols:
    rate  = df[fc].mean()
    count = int(df[fc].sum())
    bar   = '█' * int(rate * 2000)
    print(f"    {fc:14s}: {rate:>6.3%} ({count:>5,}) {bar}")

print(f"\n  Missing Values: {total_missing:,}")
print(f"  Columns: {list(df.columns)}")

  GENERATION COMPLETE

  Dataset Shape     : (50000, 19)
  Machines          : 10
  Date Range        : 2024-01-01 00:00:00 to 2024-01-06 15:19:00

  ┌──────────────────────────────────────────────┐
  │  CLASS DISTRIBUTION (IMBALANCED)             │
  ├──────────────────────────────────────────────┤
  │  Normal  (0):   48,372  (96.74%)             │
  │  Failure (1):    1,628  ( 3.26%)             │
  │  Ratio      :     29.7:1                     │
  ├──────────────────────────────────────────────┤
  │  UCI AI4I 2020 Reference: 3.4% failure rate  │
  └──────────────────────────────────────────────┘

  Failure Mode Breakdown:

    Failure_TWF   : 0.590% (  295) ███████████
    Failure_HDF   : 0.984% (  492) ███████████████████
    Failure_PWF   : 0.806% (  403) ████████████████
    Failure_OSF   : 0.812% (  406) ████████████████
    Failure_RNF   : 0.108% (   54) ██

  Missing Values: 6,949
  Columns: ['Timestamp', 'Machine_ID', 'Type', 'Shift', 'Air_Temp_K', 'Process_Temp_K', 'Rot_Spe

## 11. Imbalance Verification

In [19]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Verification
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("=" * 65)
print("  IMBALANCE VERIFICATION")
print("=" * 65)
print(f"\n  Target failure rate : {metadata['target_failure_rate']:.1%}")
print(f"  Actual failure rate : {metadata['failure_rate']:.2%}")
print(f"  Imbalance ratio     : {metadata['imbalance_ratio']:.1f}:1")
print(f"  Normal samples      : {metadata['n_normal']:,}")
print(f"  Failure samples     : {metadata['n_failures']:,}")

match = abs(metadata['failure_rate'] - metadata['target_failure_rate'])
if match < 0.01:
    print(f"\n  PASS — Within 1% of target ({match:.2%} deviation)")
else:
    print(f"\n  WARN — {match:.2%} deviation from target")

  IMBALANCE VERIFICATION

  Target failure rate : 3.5%
  Actual failure rate : 3.26%
  Imbalance ratio     : 29.7:1
  Normal samples      : 48,372
  Failure samples     : 1,628

  PASS — Within 1% of target (0.24% deviation)


In [20]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Quick preview — all records
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("Sample Data:")
df.head(10)

Sample Data:


,Timestamp,Machine_ID,Type,Shift,Air_Temp_K,Process_Temp_K,Rot_Speed_RPM,Torque_Nm,Tool_Wear_Min,Vibration_mm_s,Pressure_bar,Machine_Age_Years,Failure_TWF,Failure_HDF,Failure_PWF,Failure_OSF,Failure_RNF,Machine_Failure,Failure_Mode
0,2024-01-01,MILL-001,L,Night,295.37,305.26,3133.3,51.48,3.5,0.702,5.61,0.9,0,0,0,0,0,0,None
1,2024-01-01,MILL-009,H,Night,294.97,304.90,2794.3,47.72,5.2,0.474,5.57,6.3,0,0,0,0,0,0,None
2,2024-01-01,MILL-008,L,Night,293.01,302.97,2761.5,38.77,2.0,0.548,5.85,6.7,0,0,0,0,0,0,None
3,2024-01-01,MILL-007,H,Night,296.62,307.28,2759.1,61.91,1.1,0.473,6.38,5.0,0,0,0,0,0,0,None
4,2024-01-01,MILL-006,M,Night,292.29,301.90,2829.1,28.36,2.6,0.718,5.83,2.8,0,0,0,0,0,0,None
5,2024-01-01,MILL-005,L,Night,293.55,303.39,2985.4,54.60,3.1,0.761,5.88,5.6,0,0,0,0,0,0,None
6,2024-01-01,MILL-004,L,Night,295.96,306.51,2925.6,32.73,5.1,0.538,6.23,5.1,0,0,0,0,0,0,None
7,2024-01-01,MILL-003,L,Night,294.74,303.63,2794.7,38.74,4.3,0.545,6.13,2.7,0,0,0,0,0,0,None
8,2024-01-01,MILL-002,L,Night,294.23,304.12,2863.0,37.33,3.5,0.777,6.32,1.9,0,0,0,0,0,0,None
9,2024-01-01,MILL-010,L,Night,291.64,301.58,2982.6,48.08,4.0,0.585,6.13,3.2,0,0,0,0,0,0,None
